# 02 - Model Lab (EDA Style, Fully Visible)

This notebook keeps everything inline like your original forecast notebooks:
- explicit train/test chunk split
- explicit model blocks
- easy knobs for quick tweaking

In [3]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
!pip install statsmodels
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

try:
    from prophet import Prophet
    PROPHET_AVAILABLE = True
except Exception:
    Prophet = None
    PROPHET_AVAILABLE = False

try:
    from sklearn.linear_model import LinearRegression, Ridge
    from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
    SKLEARN_AVAILABLE = True
except Exception:
    LinearRegression = Ridge = RandomForestRegressor = ExtraTreesRegressor = GradientBoostingRegressor = None
    SKLEARN_AVAILABLE = False

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except Exception:
    XGBRegressor = None
    XGBOOST_AVAILABLE = False

try:
    from lightgbm import LGBMRegressor
    LIGHTGBM_AVAILABLE = True
except Exception:
    LGBMRegressor = None
    LIGHTGBM_AVAILABLE = False

try:
    import mlflow
    MLFLOW_AVAILABLE = True
except Exception:
    mlflow = None
    MLFLOW_AVAILABLE = False

warnings.filterwarnings("ignore")

print("PROPHET_AVAILABLE:", PROPHET_AVAILABLE)
print("SKLEARN_AVAILABLE:", SKLEARN_AVAILABLE)
print("XGBOOST_AVAILABLE:", XGBOOST_AVAILABLE)
print("LIGHTGBM_AVAILABLE:", LIGHTGBM_AVAILABLE)
print("MLFLOW_AVAILABLE:", MLFLOW_AVAILABLE)

  Using cached statsmodels-0.14.6-cp311-cp311-win_amd64.whl.metadata (9.8 kB)
  Using cached patsy-1.0.2-py2.py3-none-any.whl.metadata (3.6 kB)
Using cached statsmodels-0.14.6-cp311-cp311-win_amd64.whl (9.6 MB)
Using cached patsy-1.0.2-py2.py3-none-any.whl (233 kB)

   ---------------------------------------- 0/2 [patsy]
   ---------------------------------------- 0/2 [patsy]
   ---------------------------------------- 0/2 [patsy]
   ---------------------------------------- 0/2 [patsy]
   ---------------------------------------- 0/2 [patsy]
   ---------------------------------------- 0/2 [patsy]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- -

c:\Users\Bhavesh\Documents\Python Scripts\Jeff\Cafe\milk-dashboard\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


PROPHET_AVAILABLE: True
SKLEARN_AVAILABLE: True
XGBOOST_AVAILABLE: False
LIGHTGBM_AVAILABLE: False
MLFLOW_AVAILABLE: True


In [ ]:
ROOT = Path("..").resolve()

INPUT_MODE = "daily_csv"  # daily_csv | daily_parquet
DAILY_CSV_PATH = ROOT / "artifacts" / "data" / "daily.csv"
DAILY_PARQUET_PATH = ROOT / "artifacts" / "data" / "daily.parquet"

TEST_DAYS = 30
FUTURE_DAYS = 30

# Rolling CV chunk settings (for quick split inspection)
HOLDOUT_DAYS = 14
MIN_TRAIN_DAYS = 30
HORIZON_DAYS = 7
STEP_DAYS = 7
MAX_SPLITS = 6

# Optional logging
ENABLE_MLFLOW_LOGGING = True
TRACKING_URI = "sqlite:///../mlruns.db"
EXPERIMENT_NAME = "milk_models"

print("INPUT_MODE:", INPUT_MODE)
print("TEST_DAYS:", TEST_DAYS)

In [ ]:
if INPUT_MODE == "daily_csv":
    if not DAILY_CSV_PATH.exists():
        raise FileNotFoundError(f"Missing {DAILY_CSV_PATH}. Run 01_data_cleaning.ipynb first.")
    daily_df = pd.read_csv(DAILY_CSV_PATH)
    source_info = str(DAILY_CSV_PATH)
elif INPUT_MODE == "daily_parquet":
    if not DAILY_PARQUET_PATH.exists():
        raise FileNotFoundError(f"Missing {DAILY_PARQUET_PATH}. Run 01_data_cleaning.ipynb first.")
    daily_df = pd.read_parquet(DAILY_PARQUET_PATH)
    source_info = str(DAILY_PARQUET_PATH)
else:
    raise ValueError(f"Unknown INPUT_MODE: {INPUT_MODE}")

daily_df["date"] = pd.to_datetime(daily_df["date"], errors="coerce")
if "gallons" not in daily_df.columns:
    if "total_milk_oz" not in daily_df.columns:
        raise ValueError("Need either 'gallons' or 'total_milk_oz' in input")
    daily_df["gallons"] = pd.to_numeric(daily_df["total_milk_oz"], errors="coerce") / 128.0

daily_df["gallons"] = pd.to_numeric(daily_df["gallons"], errors="coerce")
daily_df = daily_df.dropna(subset=["date", "gallons"]).sort_values("date").reset_index(drop=True)

print("source:", source_info)
print("rows:", len(daily_df))
print("date range:", daily_df["date"].min(), "->", daily_df["date"].max())
print("avg gallons:", round(float(daily_df["gallons"].mean()), 3))
daily_df.head()

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(daily_df["date"], daily_df["gallons"], linewidth=1.8)
plt.title("Daily Milk Usage (Gallons)")
plt.xlabel("Date")
plt.ylabel("Gallons")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
if len(daily_df) <= TEST_DAYS:
    raise ValueError("Not enough rows for TEST_DAYS split")

train_df = daily_df.iloc[:-TEST_DAYS].copy()
test_df = daily_df.iloc[-TEST_DAYS:].copy()

print("TRAIN rows:", len(train_df), "| TEST rows:", len(test_df))
print("TRAIN range:", train_df["date"].min(), "->", train_df["date"].max())
print("TEST range:", test_df["date"].min(), "->", test_df["date"].max())


def build_rolling_cv_chunks(df, min_train_days=30, horizon_days=7, step_days=7, max_splits=5):
    d = df.sort_values("date").reset_index(drop=True)
    splits = []
    end_idx = min_train_days
    split_idx = 1
    while end_idx + horizon_days <= len(d):
        split_train = d.iloc[:end_idx].reset_index(drop=True)
        split_test = d.iloc[end_idx:end_idx + horizon_days].reset_index(drop=True)
        splits.append({"name": f"cv_{split_idx}", "train": split_train, "test": split_test})
        split_idx += 1
        end_idx += step_days
    return splits[-max_splits:]


cv_chunks = build_rolling_cv_chunks(
    train_df,
    min_train_days=MIN_TRAIN_DAYS,
    horizon_days=HORIZON_DAYS,
    step_days=STEP_DAYS,
    max_splits=MAX_SPLITS,
)

cv_summary = []
for s in cv_chunks:
    cv_summary.append(
        {
            "split": s["name"],
            "train_rows": len(s["train"]),
            "test_rows": len(s["test"]),
            "train_end": str(s["train"]["date"].max()),
            "test_end": str(s["test"]["date"].max()),
        }
    )

pd.DataFrame(cv_summary)

In [ ]:
def rmse(actual, pred):
    actual = np.asarray(actual, dtype=float)
    pred = np.asarray(pred, dtype=float)
    return float(np.sqrt(np.mean((actual - pred) ** 2)))


def mae(actual, pred):
    actual = np.asarray(actual, dtype=float)
    pred = np.asarray(pred, dtype=float)
    return float(np.mean(np.abs(actual - pred)))


def mape(actual, pred):
    actual = np.asarray(actual, dtype=float)
    pred = np.asarray(pred, dtype=float)
    mask = actual != 0
    if np.any(mask):
        return float(np.mean(np.abs((actual[mask] - pred[mask]) / actual[mask])) * 100.0)
    return float("nan")


def accuracy_from_mape(mape_value):
    if np.isnan(mape_value):
        return float("nan")
    return float(100.0 - mape_value)


def bias(actual, pred):
    actual = np.asarray(actual, dtype=float)
    pred = np.asarray(pred, dtype=float)
    return float(np.mean(pred - actual))


def metric_dict(actual, pred):
    mape_v = mape(actual, pred)
    return {
        "RMSE": rmse(actual, pred),
        "MAE": mae(actual, pred),
        "MAPE": mape_v,
        "Accuracy": accuracy_from_mape(mape_v),
        "Bias": bias(actual, pred),
    }

In [ ]:
ts_predictions = {}

# ARIMA
try:
    arima_model = ARIMA(train_df["gallons"], order=(2, 1, 2))
    arima_fit = arima_model.fit()
    pred = arima_fit.forecast(steps=TEST_DAYS)
    ts_predictions["ARIMA"] = pd.Series(np.maximum(pred.values, 0.0), index=test_df["date"].values)
    print("OK: ARIMA")
except Exception as e:
    print("SKIP: ARIMA ->", type(e).__name__, str(e))

# SARIMA
try:
    sarima_model = SARIMAX(
        train_df["gallons"],
        order=(2, 1, 2),
        seasonal_order=(1, 1, 1, 7),
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    sarima_fit = sarima_model.fit(disp=False)
    pred = sarima_fit.get_forecast(steps=TEST_DAYS).predicted_mean
    ts_predictions["SARIMA"] = pd.Series(np.maximum(pred.values, 0.0), index=test_df["date"].values)
    print("OK: SARIMA")
except Exception as e:
    print("SKIP: SARIMA ->", type(e).__name__, str(e))

# Prophet
if PROPHET_AVAILABLE:
    try:
        prophet_train = train_df[["date", "gallons"]].rename(columns={"date": "ds", "gallons": "y"})
        prophet_model = Prophet(
            seasonality_mode="multiplicative",
            weekly_seasonality=True,
            yearly_seasonality=False,
            daily_seasonality=False,
            changepoint_prior_scale=0.05,
            seasonality_prior_scale=1.0,
        )
        prophet_model.fit(prophet_train)
        future = prophet_model.make_future_dataframe(periods=TEST_DAYS, freq="D")
        forecast = prophet_model.predict(future).tail(TEST_DAYS)
        pred = np.maximum(forecast["yhat"].values, 0.0)
        ts_predictions["Prophet"] = pd.Series(pred, index=test_df["date"].values)
        print("OK: Prophet")
    except Exception as e:
        print("SKIP: Prophet ->", type(e).__name__, str(e))
else:
    print("SKIP: Prophet -> package not installed")

In [ ]:
reg_df = daily_df[["date", "gallons"]].copy().sort_values("date")

# lag features
for lag in [1, 2, 3, 7, 14, 21, 28]:
    reg_df[f"lag_{lag}"] = reg_df["gallons"].shift(lag)

# rolling features
reg_df["rolling_mean_7"] = reg_df["gallons"].rolling(7).mean()
reg_df["rolling_std_7"] = reg_df["gallons"].rolling(7).std()
reg_df["rolling_mean_14"] = reg_df["gallons"].rolling(14).mean()

# calendar features
reg_df["day_of_week"] = reg_df["date"].dt.dayofweek
reg_df["is_weekend"] = (reg_df["day_of_week"] >= 5).astype(int)
reg_df["month"] = reg_df["date"].dt.month

reg_df = reg_df.dropna().reset_index(drop=True)

if len(reg_df) <= TEST_DAYS:
    raise ValueError("Not enough rows in feature table for TEST_DAYS")

train_reg = reg_df.iloc[:-TEST_DAYS].copy()
test_reg = reg_df.iloc[-TEST_DAYS:].copy()

feature_cols = [c for c in reg_df.columns if c not in ["date", "gallons"]]

X_train = train_reg[feature_cols]
y_train = train_reg["gallons"]
X_test = test_reg[feature_cols]
y_test_reg = test_reg["gallons"]

print("Regression features:", len(feature_cols))
print(feature_cols)

In [ ]:
reg_models = {}
reg_predictions = {}

if SKLEARN_AVAILABLE:
    reg_models["LinearRegression"] = LinearRegression()
    reg_models["Ridge"] = Ridge(alpha=1.0)
    reg_models["RandomForest"] = RandomForestRegressor(
        n_estimators=500,
        max_depth=8,
        random_state=42,
        n_jobs=-1,
    )
    reg_models["ExtraTrees"] = ExtraTreesRegressor(
        n_estimators=500,
        max_depth=8,
        random_state=42,
        n_jobs=-1,
    )
    reg_models["GradientBoosting"] = GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42,
    )

if XGBOOST_AVAILABLE:
    reg_models["XGBoost"] = XGBRegressor(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42,
        objective="reg:squarederror",
    )

if LIGHTGBM_AVAILABLE:
    reg_models["LightGBM"] = LGBMRegressor(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=6,
        random_state=42,
    )

for name, model in reg_models.items():
    try:
        model.fit(X_train, y_train)
        pred = np.maximum(model.predict(X_test), 0.0)
        reg_predictions[name] = pd.Series(pred, index=test_reg["date"].values)
        print("OK:", name)
    except Exception as e:
        print("SKIP:", name, "->", type(e).__name__, str(e))

In [ ]:
results = {}

# Time-series models
for name, pred_series in ts_predictions.items():
    y_true = test_df["gallons"].values
    y_pred = pred_series.values
    results[name] = metric_dict(y_true, y_pred)

# Regression models
for name, pred_series in reg_predictions.items():
    y_true = y_test_reg.values
    y_pred = pred_series.values
    results[name] = metric_dict(y_true, y_pred)

comparison = pd.DataFrame(results).T
comparison = comparison[["RMSE", "MAE", "MAPE", "Accuracy", "Bias"]]
comparison = comparison.sort_values(["RMSE", "MAE"]).reset_index().rename(columns={"index": "Model"})

print("MODEL PERFORMANCE COMPARISON")
comparison

In [ ]:
# Plot top 5 models on holdout
top_models = comparison["Model"].head(5).tolist()

# pick matching holdout date index for each model family
ts_index = test_df["date"].values
reg_index = test_reg["date"].values

for model_name in top_models:
    plt.figure(figsize=(11, 4))
    if model_name in ts_predictions:
        idx = ts_index
        actual = test_df["gallons"].values
        pred = ts_predictions[model_name].values
    else:
        idx = reg_index
        actual = y_test_reg.values
        pred = reg_predictions[model_name].values

    plt.plot(idx, actual, label="Actual", linewidth=2)
    plt.plot(idx, pred, label=model_name, linestyle="--", linewidth=2)
    plt.title(f"{model_name} vs Actual")
    plt.xlabel("Date")
    plt.ylabel("Gallons")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
best_model = comparison.iloc[0]["Model"]
print("Best model:", best_model)

# Future forecast from best model (simple and tweakable)
if best_model == "ARIMA":
    fit = ARIMA(daily_df["gallons"], order=(2, 1, 2)).fit()
    future_vals = np.maximum(fit.forecast(steps=FUTURE_DAYS).values, 0.0)
    future_dates = pd.date_range(daily_df["date"].max() + pd.Timedelta(days=1), periods=FUTURE_DAYS, freq="D")
    future_forecast = pd.DataFrame({"date": future_dates, "forecast_gallons": future_vals})

elif best_model == "SARIMA":
    fit = SARIMAX(
        daily_df["gallons"],
        order=(2, 1, 2),
        seasonal_order=(1, 1, 1, 7),
        enforce_stationarity=False,
        enforce_invertibility=False,
    ).fit(disp=False)
    future_vals = np.maximum(fit.get_forecast(steps=FUTURE_DAYS).predicted_mean.values, 0.0)
    future_dates = pd.date_range(daily_df["date"].max() + pd.Timedelta(days=1), periods=FUTURE_DAYS, freq="D")
    future_forecast = pd.DataFrame({"date": future_dates, "forecast_gallons": future_vals})

elif best_model == "Prophet" and PROPHET_AVAILABLE:
    prophet_full = daily_df[["date", "gallons"]].rename(columns={"date": "ds", "gallons": "y"})
    m = Prophet(
        seasonality_mode="multiplicative",
        weekly_seasonality=True,
        yearly_seasonality=False,
        daily_seasonality=False,
        changepoint_prior_scale=0.05,
        seasonality_prior_scale=1.0,
    )
    m.fit(prophet_full)
    fc = m.predict(m.make_future_dataframe(periods=FUTURE_DAYS, freq="D")).tail(FUTURE_DAYS)
    future_forecast = fc[["ds", "yhat"]].rename(columns={"ds": "date", "yhat": "forecast_gallons"})
    future_forecast["forecast_gallons"] = np.maximum(future_forecast["forecast_gallons"], 0.0)

else:
    # Regression-style recursive fallback using RandomForest if available, else moving average
    if SKLEARN_AVAILABLE:
        fallback = RandomForestRegressor(n_estimators=500, max_depth=8, random_state=42, n_jobs=-1)
        fallback.fit(X_train, y_train)

        hist = reg_df[["date", "gallons"]].copy().sort_values("date").reset_index(drop=True)
        rows = []
        for _ in range(FUTURE_DAYS):
            next_date = hist["date"].iloc[-1] + pd.Timedelta(days=1)
            temp = hist.copy()
            temp.loc[len(temp)] = [next_date, np.nan]

            for lag in [1, 2, 3, 7, 14, 21, 28]:
                temp[f"lag_{lag}"] = temp["gallons"].shift(lag)
            temp["rolling_mean_7"] = temp["gallons"].rolling(7).mean()
            temp["rolling_std_7"] = temp["gallons"].rolling(7).std()
            temp["rolling_mean_14"] = temp["gallons"].rolling(14).mean()
            temp["day_of_week"] = temp["date"].dt.dayofweek
            temp["is_weekend"] = (temp["day_of_week"] >= 5).astype(int)
            temp["month"] = temp["date"].dt.month

            feat = temp.iloc[[-1]][feature_cols].fillna(method="ffill").fillna(method="bfill")
            pred = max(float(fallback.predict(feat)[0]), 0.0)

            rows.append({"date": next_date, "forecast_gallons": pred})
            hist.loc[len(hist)] = [next_date, pred]

        future_forecast = pd.DataFrame(rows)
    else:
        mean_val = float(daily_df["gallons"].tail(7).mean())
        future_dates = pd.date_range(daily_df["date"].max() + pd.Timedelta(days=1), periods=FUTURE_DAYS, freq="D")
        future_forecast = pd.DataFrame({"date": future_dates, "forecast_gallons": [mean_val] * FUTURE_DAYS})

future_forecast.head()

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(future_forecast["date"], future_forecast["forecast_gallons"], linewidth=2)
plt.title(f"Next {FUTURE_DAYS} Days Forecast ({best_model})")
plt.xlabel("Date")
plt.ylabel("Forecast Gallons")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
run_info = None

if ENABLE_MLFLOW_LOGGING and MLFLOW_AVAILABLE:
    try:
        mlflow.set_tracking_uri(TRACKING_URI)
        exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
        experiment_id = exp.experiment_id if exp else mlflow.create_experiment(EXPERIMENT_NAME)

        with mlflow.start_run(experiment_id=experiment_id):
            mlflow.log_param("input_mode", INPUT_MODE)
            mlflow.log_param("test_days", TEST_DAYS)
            mlflow.log_param("best_model", best_model)
            mlflow.log_metrics({f"best_{k.lower()}": float(v) for k, v in comparison.iloc[0].drop("Model").items()})

            tmp_cmp = ROOT / "artifacts" / "data" / "model_lab_comparison.csv"
            tmp_fc = ROOT / "artifacts" / "data" / "model_lab_future_forecast.csv"
            tmp_cmp.parent.mkdir(parents=True, exist_ok=True)
            comparison.to_csv(tmp_cmp, index=False)
            future_forecast.to_csv(tmp_fc, index=False)

            mlflow.log_artifact(str(tmp_cmp), artifact_path="tables")
            mlflow.log_artifact(str(tmp_fc), artifact_path="predictions")

            run = mlflow.active_run()
            run_info = {
                "run_id": run.info.run_id if run else None,
                "experiment": EXPERIMENT_NAME,
            }
        print("MLflow run:", run_info)
    except Exception as e:
        print("MLflow logging skipped:", type(e).__name__, str(e))
else:
    print("MLflow disabled or not installed")